In [1]:
# 单元格 2：导入库
# 说明：requests 负责请求智联前端 JSON 接口，pandas 负责表格处理，SQLAlchemy 负责写入 MySQL。

from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import hashlib
import json
import os
import random
import re
import time
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import quote_plus, urlencode

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from sqlalchemy import create_engine, text
from tqdm.auto import tqdm


c:\Users\10967\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 单元格 3：自定义搜索参数

最常改的地方都在这里：关键词、城市列表、页数、暂停时间。比如要抓“后端开发”，把 `KEYWORD` 改成 `后端开发`；如果只想抓重庆，把 `CITY_LIST` 改成 `['重庆']`。


In [2]:
# 单元格 3：自定义搜索参数
# 你主要改这里：KEYWORD、CITY_LIST、START_PAGE、MAX_PAGES。

KEYWORD = '数据开发'        # 搜索关键词，例如：'Python'、'数据分析'、'运维'

# 按网页上方城市入口批量抓取；每个城市都会抓 START_PAGE 到 START_PAGE + MAX_PAGES - 1。
# CITY_LIST = [
#     '全国'
# ]
CITY_LIST = [
    '北京', '上海', '广州', '深圳', '天津', '武汉', '西安', '成都',
    '大连', '长春', '沈阳', '南京', '济南', '青岛', '杭州', '苏州',
    '无锡', '宁波', '重庆', '郑州', '长沙', '福州', '厦门', '哈尔滨'
]

# 单城市调试：如果只想抓一个城市，把 CITY_LIST 改成 ['重庆'] 即可。
CITY = CITY_LIST[0]
CITY_CODE = None          # 单城市手动代码；批量城市通常保持 None，自动从 CITY_CODE_MAP 取

START_PAGE = 1            # 每个城市的起始页
MAX_PAGES = 5             # 每个城市最多抓取多少页；当前需求是每个城市 1-5 页
PAGE_SIZE = 20            # 每页条数，智联当前接口常用 20
EMPTY_PAGE_STOP = 2       # 连续多少个空页后停止，避免某城市没有足够页数时一直请求
PAGE_SLEEP_RANGE = (0.4, 1.0)  # 每页之间随机暂停；如果频繁触发验证，可调大
MAX_CITY_WORKERS = 4      # 城市级并发数；调小更稳，调大更快但更容易触发安全验证
REQUEST_TIMEOUT = 25      # 单次请求超时时间（秒）
REQUEST_RETRIES = 4       # 单页请求失败后的最大重试次数
RETRY_BACKOFF_BASE = 1.0  # 重试等待的基础秒数，会逐次递增
RETRY_SLEEP_RANGE = (0.3, 1.0)  # 每次重试额外增加的随机等待秒数

# MySQL 入库配置。密码优先读取环境变量；未设置时使用本机训练库密码。
MYSQL_HOST = '127.0.0.1'  # MySQL 服务地址
MYSQL_PORT = 3306  # MySQL 服务端口
MYSQL_USER = 'root'  # MySQL 登录用户名
MYSQL_PASSWORD = os.environ.get('SHIXUN_MYSQL_PASSWORD', '1472580369@Lzh')  # MySQL 密码，优先读取环境变量
MYSQL_DATABASE = 'shixun'  # 项目使用的数据库名
RAW_JOB_TABLE = 'raw_job_info'  # 原始岗位信息表名
RAW_COMPANY_TABLE = 'raw_company_info'  # 原始公司信息表名
RAW_HR_TABLE = 'raw_hr_status_info'  # 原始 HR 状态表名
CRAWLER_LOG_TABLE = 'crawler_log'  # 爬虫批次日志表名
KEEP_RECENT_BATCHES = 5  # 同一关键词最多保留的最近批次数


def locate_project_root():
    """
    Input:
    - 无。
    Output:
    - Path，项目根目录。
    Function:
    - 定位当前项目根目录。
    """
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / '爬虫').is_dir() and (base / '数据清洗上传HDFS').is_dir():
            return base
    raise FileNotFoundError('没有找到项目根目录：当前项目根目录应同时包含“爬虫”和“数据清洗上传HDFS”。')


PROJECT_ROOT = locate_project_root()  # 项目根目录，供后续定位数据和代码目录使用

# 常用城市代码。若目标城市不在这里，在这里补充。
CITY_CODE_MAP = {
    '全国': '489',
    '北京': '530', '上海': '538', '天津': '531', '重庆': '551',
    '广州': '763', '深圳': '765', '武汉': '736', '西安': '854',
    '成都': '801', '大连': '600', '长春': '613', '沈阳': '599',
    '南京': '635', '济南': '702', '青岛': '703', '杭州': '653',
    '苏州': '639', '无锡': '636', '宁波': '654', '郑州': '719',
    '长沙': '749', '福州': '681', '厦门': '682', '哈尔滨': '622'
}

# 额外筛选条件。会直接合并进 JSON 请求体。
# order=4 是当前接口常见的排序值；如果后续失效，可删掉或改成接口支持的其他值。
EXTRA_PARAMS = {
    'order': 4,
}


## 单元格 4：请求头与安全验证检测

智联搜索页 HTML 当前容易返回安全验证页。本 notebook 改为请求前端 JSON 接口，并保留安全验证检测，避免把验证页误当成岗位数据。


In [3]:
# 单元格 4：请求头与安全验证检测
# 说明：搜索页 HTML 现在经常返回安全验证；这里改为请求智联前端 JSON 接口。

SEARCH_API_URL = 'https://fe-api.zhaopin.com/c/i/search/positions'  # 智联岗位搜索 JSON 接口地址

# 请求头参数：模拟浏览器前端接口请求，降低被接口拒绝的概率。
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/125.0.0.0 Safari/537.36'
    ),
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
    'Content-Type': 'application/json;charset=UTF-8',
    'Origin': 'https://www.zhaopin.com',
    'Referer': 'https://www.zhaopin.com/',
    'Connection': 'keep-alive',
    'x-zp-page-code': '4019',
    'x-zp-platform': '13',
    'x-zp-business-system': '1',
}

# 安全验证特征：响应内容命中这些文本时，说明可能被网站验证拦截。
SECURITY_MARKERS = (
    'Security Verification',
    'TencentEOCaptcha',
    'EO-Bot-Captcha-Token',
    '正在验证连接安全性',
    'Protected by Tencent Cloud EdgeOne',
)


def make_request_session() -> requests.Session:
    """
    Input:
    - 无。
    Output:
    - requests.Session，会话对象。
    Function:
    - 创建带请求头和连接池的 requests 会话。
    """
    new_session = requests.Session()
    new_session.headers.update(HEADERS)
    worker_count = max(1, int(globals().get('MAX_CITY_WORKERS', 1)))
    pool_size = max(8, worker_count * 4)
    adapter = HTTPAdapter(pool_connections=pool_size, pool_maxsize=pool_size, max_retries=0)
    new_session.mount('https://', adapter)
    new_session.mount('http://', adapter)
    return new_session


session = make_request_session()


def reset_request_session() -> None:
    """
    Input:
    - 无。
    Output:
    - None。
    Function:
    - 关闭旧会话并重建请求会话。
    """
    global session
    try:
        session.close()
    except Exception:
        pass
    session = make_request_session()


def looks_like_security_verification(text: str) -> bool:
    """
    Input:
    - text: 需要判断、清理或解析的文本。
    Output:
    - bool，是否命中安全验证特征。
    Function:
    - 判断响应内容是否像安全验证页面。
    """
    return any(marker in text for marker in SECURITY_MARKERS)

## 单元格 5：城市代码、展示 URL 与接口请求体构造

`build_search_page_url` 用于记录和 Referer；真正取数的是 `build_search_payload` 构造出的 JSON 请求体。


In [4]:
# 单元格 5：城市代码、展示 URL 与接口请求体构造
# 说明：build_search_page_url 只用于记录和 Referer；真正取数用 build_search_payload。


def resolve_city_code(city: str, city_code: Optional[str] = None) -> str:
    """
    Input:
    - city: 搜索城市名称。
    - city_code: 智联城市代码；为空时从城市映射表查找。
    Output:
    - str，城市代码。
    Function:
    - 确定接口请求使用的城市代码。
    """
    if city_code not in (None, ''):
        return str(city_code)
    if city in CITY_CODE_MAP:
        return CITY_CODE_MAP[city]
    raise ValueError(f'城市 {city!r} 不在 CITY_CODE_MAP 中，请手动设置 CITY_CODE。')


def build_search_page_url(
    keyword: str,
    city_code: str,
    page: int,
    extra_params: Optional[Dict[str, Any]] = None,
) -> str:
    """
    Input:
    - keyword: 搜索关键词，也是后续批次筛选条件。
    - city_code: 智联城市代码；为空时从城市映射表查找。
    - page: 请求页码。
    - extra_params: 额外接口筛选参数。
    Output:
    - str，搜索页 URL。
    Function:
    - 构造用于日志和 Referer 的搜索页 URL。
    """
    params = {'kw': keyword, 'p': str(page)}
    if city_code:
        params['jl'] = str(city_code)
    if extra_params:
        for key, value in extra_params.items():
            if value not in (None, '') and key not in {'S_SOU_FULL_INDEX', 'S_SOU_WORK_CITY', 'pageIndex', 'pageSize'}:
                params[key] = value
    return 'https://www.zhaopin.com/sou/?' + urlencode(params)


def build_api_query_params() -> Dict[str, str]:
    """
    Input:
    - 无。
    Output:
    - dict，请求参数。
    Function:
    - 构造接口 URL 上的动态请求参数。
    """
    return {
        '_v': f'{random.random():.8f}',
        'x-zp-page-request-id': f'{int(time.time() * 1000)}-{random.randint(100000, 999999)}',
        'x-zp-client-id': str(uuid.uuid4()),
    }


def build_search_payload(
    keyword: str,
    city_code: str,
    page: int,
    page_size: int = 20,
    extra_params: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Input:
    - keyword: 搜索关键词，也是后续批次筛选条件。
    - city_code: 智联城市代码；为空时从城市映射表查找。
    - page: 请求页码。
    - page_size: 每页岗位数量。
    - extra_params: 额外接口筛选参数。
    Output:
    - dict，请求体。
    Function:
    - 构造智联岗位搜索 JSON 请求体。
    """
    payload: Dict[str, Any] = {
        'S_SOU_FULL_INDEX': keyword,
        'pageIndex': int(page),
        'pageSize': int(page_size),
        'anonymous': 1,
        'eventScenario': 'pcSearchedSouSearch',
        'platform': 13,
        'version': '0.0.0',
        'order': 4,
    }
    if city_code:
        payload['S_SOU_WORK_CITY'] = str(city_code)
    if extra_params:
        payload.update({k: v for k, v in extra_params.items() if v not in (None, '')})
    return payload


# 先打印一个示例 URL 和请求体，确认关键词和城市参数是否正确。
resolved_city_code = resolve_city_code(CITY, CITY_CODE)
example_url = build_search_page_url(KEYWORD, resolved_city_code, START_PAGE, EXTRA_PARAMS)
example_payload = build_search_payload(KEYWORD, resolved_city_code, START_PAGE, PAGE_SIZE, EXTRA_PARAMS)
print(example_url)
print(json.dumps(example_payload, ensure_ascii=False, indent=2))


https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=530&order=4
{
  "S_SOU_FULL_INDEX": "数据开发",
  "pageIndex": 1,
  "pageSize": 20,
  "anonymous": 1,
  "eventScenario": "pcSearchedSouSearch",
  "platform": 13,
  "version": "0.0.0",
  "order": 4,
  "S_SOU_WORK_CITY": "530"
}


## 单元格 6：JSON 接口请求函数

这一格请求 `https://fe-api.zhaopin.com/c/i/search/positions`，并检查是否返回安全验证或异常 JSON。


In [5]:
# 单元格 6：JSON 接口请求函数
# 说明：搜索页 HTML 会被安全验证拦截；这里直接请求前端 JSON 接口并返回 data。

RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}  # 这些 HTTP 状态码会触发重试
RETRYABLE_REQUEST_EXCEPTIONS = (  # 这些网络异常会触发重试

    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    requests.exceptions.ChunkedEncodingError,
)


class PageRequestError(RuntimeError):
    """单页接口请求多次重试后仍失败。"""


def retry_wait_seconds(attempt: int) -> float:
    """
    Input:
    - attempt: 当前重试次数，从 1 开始。
    Output:
    - float，等待秒数。
    Function:
    - 计算重试等待时间。
    """
    base = RETRY_BACKOFF_BASE * (2 ** max(attempt - 1, 0))
    jitter = random.uniform(*RETRY_SLEEP_RANGE)
    return min(base + jitter, 30.0)


def fetch_position_page(
    keyword: str,
    city_code: str,
    page: int,
    page_size: int = 20,
    extra_params: Optional[Dict[str, Any]] = None,
    timeout: int = 25,
    retries: Optional[int] = None,
    request_session: Optional[requests.Session] = None,
) -> Tuple[Dict[str, Any], Dict[str, Any], str]:
    """
    Input:
    - keyword: 搜索关键词，也是后续批次筛选条件。
    - city_code: 智联城市代码；为空时从城市映射表查找。
    - page: 请求页码。
    - page_size: 每页岗位数量。
    - extra_params: 额外接口筛选参数。
    - timeout: 单次请求超时时间，单位秒。
    - retries: 最大重试次数；为空时使用全局默认值。
    - request_session: requests 会话对象；为空时使用全局会话。
    Output:
    - tuple，接口 data、请求 payload、请求 URL。
    Function:
    - 请求一页岗位 JSON，并处理重试和安全验证。
    """
    payload = build_search_payload(keyword, city_code, page, page_size, extra_params)
    page_url = build_search_page_url(keyword, city_code, page, extra_params)
    headers = dict(HEADERS)
    headers['Referer'] = page_url
    max_retries = REQUEST_RETRIES if retries is None else int(retries)
    last_error: Optional[BaseException] = None
    active_session = request_session or session

    for attempt in range(1, max_retries + 1):
        api_params = build_api_query_params()
        try:
            resp = active_session.post(
                SEARCH_API_URL,
                params=api_params,
                headers=headers,
                data=json.dumps(payload, ensure_ascii=False).encode('utf-8'),
                timeout=timeout,
            )

            if resp.status_code in RETRYABLE_STATUS_CODES:
                raise requests.exceptions.HTTPError(
                    f'HTTP {resp.status_code}: {resp.text[:200]!r}',
                    response=resp,
                )

            resp.raise_for_status()
            text = resp.text
            if looks_like_security_verification(text):
                raise RuntimeError('接口返回了网站安全验证页。请降低频率，稍后重试，或在浏览器中完成验证后再运行。')

            try:
                result = resp.json()
            except ValueError as exc:
                raise RuntimeError(f'接口没有返回 JSON，前 300 字符：{text[:300]!r}') from exc

            if result.get('code') != 200 or result.get('apiCode') not in (None, 200):
                raise RuntimeError(f'接口返回异常：{json.dumps(result, ensure_ascii=False)[:500]}')

            data = result.get('data') or {}
            if data.get('isVerification'):
                raise RuntimeError('接口提示需要安全验证，未继续抓取。')
            if not isinstance(data.get('list', []), list):
                raise RuntimeError(f'接口 data.list 不是列表：{json.dumps(data, ensure_ascii=False)[:500]}')

            return data, payload, resp.url

        except RETRYABLE_REQUEST_EXCEPTIONS as exc:
            last_error = exc
            if request_session is None:
                reset_request_session()
                active_session = session
        except requests.exceptions.HTTPError as exc:
            last_error = exc
            response = getattr(exc, 'response', None)
            status_code = getattr(response, 'status_code', None)
            if status_code not in RETRYABLE_STATUS_CODES:
                raise
            if request_session is None:
                reset_request_session()
                active_session = session

        if attempt < max_retries:
            wait_seconds = retry_wait_seconds(attempt)
            print(f'第 {page} 页请求失败，{wait_seconds:.1f} 秒后重试 {attempt}/{max_retries}：{last_error!r}')
            time.sleep(wait_seconds)

    raise PageRequestError(f'第 {page} 页连续 {max_retries} 次请求失败：{last_error!r}') from last_error

## 单元格 7：浏览器备用说明

当前版本不再使用 Playwright 抓 HTML。最后保留一个空的关闭函数，用于统一结束流程。


In [6]:
# 单元格 7：浏览器备用说明
# 说明：当前版本不再依赖 Playwright 解析页面 HTML；保留关闭函数用于统一结束流程。


def close_playwright_browser():
    """
    Input:
    - 无。
    Output:
    - None。
    Function:
    - 保留浏览器关闭入口；当前接口版不启动浏览器。
    """
    return None


## 单元格 8：解析接口返回数据

职位列表来自接口返回的 `data.list`，总数来自 `data.count`，是否最后一页来自 `data.isEndPage`。


In [7]:
# 单元格 8：解析接口返回数据
# 说明：职位列表在接口返回的 data.list 中，count 是接口给出的结果数量，isEndPage 表示是否最后一页。


def extract_position_page_data(data: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], int, bool]:
    """
    Input:
    - data: 接口返回的 data 字典。
    Output:
    - tuple，岗位列表、总数、是否末页。
    Function:
    - 从接口 data 中提取岗位列表、总数和是否末页。
    """
    jobs = data.get('list') or []
    count = int(data.get('count') or 0)
    is_end_page = bool(data.get('isEndPage'))
    return jobs, count, is_end_page


## 单元格 9：基础清洗工具

后面整理字段时，会遇到列表、字典、空值、重复标签等情况。本单元格放通用小函数，避免后面的字段整理代码太乱。


In [8]:
# 单元格 9：基础清洗工具
# 说明：这些函数只做文本清洗、路径取值、列表合并，不涉及具体业务字段。


def get_path(obj: Any, path: List[str], default: Any = '') -> Any:
    """
    Input:
    - obj: 需要读取的字典对象。
    - path: 多层字段路径。
    - default: 取值失败时返回的默认值。
    Output:
    - Any，取到的值或默认值。
    Function:
    - 按路径从多层字典中安全取值。
    """
    cur = obj
    for key in path:
        if isinstance(cur, dict) and key in cur:
            cur = cur[key]
        else:
            return default
    return default if cur is None else cur


def clean_text(value: Any) -> str:
    """
    Input:
    - value: 需要清洗、格式化或转义的输入值。
    Output:
    - str，清洗后的文本。
    Function:
    - 清洗文本空白并统一转成字符串。
    """
    if value is None:
        return ''
    if isinstance(value, str):
        return ' '.join(value.split())
    return str(value)


def unique_join(values: List[Any], sep: str = ' | ') -> str:
    """
    Input:
    - values: 参与拼接或签名的一组值。
    - sep: 拼接字符串时使用的分隔符。
    Output:
    - str，拼接后的文本。
    Function:
    - 去空去重后拼接列表文本。
    """
    seen = set()
    result = []
    for value in values:
        text = clean_text(value)
        if not text or text in seen:
            continue
        seen.add(text)
        result.append(text)
    return sep.join(result)


def parse_json_field(value: Any) -> Dict[str, Any]:
    """
    Input:
    - value: 需要清洗、格式化或转义的输入值。
    Output:
    - dict，解析结果；失败时为空字典。
    Function:
    - 解析字段中嵌套的 JSON 字符串。
    """
    if not value or not isinstance(value, str):
        return {}
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return {}


## 单元格 10：标签提取工具

岗位标签来源很多：`showSkillTags`、`jobSkillTags`、福利、职位关键词、职位详情里的标签等。本单元格把这些不同格式统一成文本列表。


In [9]:
# 单元格 10：标签提取工具
# 说明：尽可能把岗位相关标签都保留下来，后面你可以自己筛选需要的列。


def list_values(
    items: Any,
    keys: Tuple[str, ...] = ('name', 'value', 'tag', 'itemValue', 'title', 'text', 'label', 'description'),
) -> List[str]:
    """
    Input:
    - items: 需要提取标签的原始列表、字典或普通值。
    - keys: 从字典中读取标签时尝试的候选字段名。
    Output:
    - list[str]，标签文本列表。
    Function:
    - 从多种标签结构中提取标签文本。
    """
    if not items:
        return []
    if isinstance(items, dict):
        items = [items]
    if isinstance(items, (str, int, float)):
        return [clean_text(items)]

    values = []
    for item in items:
        if isinstance(item, dict):
            for key in keys:
                if item.get(key) not in (None, ''):
                    values.append(item[key])
                    break
        else:
            values.append(item)
    return [clean_text(v) for v in values if clean_text(v)]


def key_value_items(items: Any, name_key: str = 'name', value_key: str = 'value') -> List[str]:
    """
    Input:
    - items: 需要提取标签的原始列表、字典或普通值。
    - name_key: 键值结构中的名称字段名。
    - value_key: 键值结构中的值字段名。
    Output:
    - list[str]，键值标签列表。
    Function:
    - 把 name/value 结构转为可读标签。
    """
    if not items:
        return []

    values = []
    for item in items:
        if not isinstance(item, dict):
            values.append(clean_text(item))
            continue

        name = clean_text(item.get(name_key, ''))
        value = clean_text(item.get(value_key, ''))
        if name and value and name != value:
            values.append(f'{name}:{value}')
        elif name:
            values.append(name)
        elif value:
            values.append(value)
    return values


def collect_all_tags(job: Dict[str, Any]) -> str:
    """
    Input:
    - job: 单条岗位原始 JSON。
    Output:
    - str，标签汇总文本。
    Function:
    - 汇总单个岗位中的所有标签。
    """
    jd = job.get('jobDetailData') or {}
    custom = jd.get('customAttributeInfo') or {}
    desc = get_path(jd, ['position', 'desc'], {})

    tags = []
    tags += list_values(job.get('jobSkillTags'), ('name', 'value', 'tag'))
    tags += list_values(job.get('skillLabel'), ('value', 'name', 'tag'))
    tags += list_values(job.get('showSkillTags'), ('tag', 'name', 'value'))
    tags += list_values(get_path(job, ['jobKeyword', 'keywords'], []), ('itemValue', 'name', 'tag', 'value'))
    tags += list_values(desc.get('labels'))
    tags += list_values(desc.get('welfareLabel'))
    tags += list_values(desc.get('welfareTags'))
    tags += list_values(job.get('welfareLabel'))
    tags += list_values(custom.get('reportItems'))
    tags += key_value_items(custom.get('welfareItems'))
    tags += key_value_items(custom.get('workTimeItems'))
    tags += list_values(job.get('searchTagList'))
    tags += list_values(job.get('commercialLabel'))
    tags += list_values(job.get('orgCommercialTags'))
    tags += list_values(job.get('companyScaleTypeTagsNew'))

    if job.get('tagABC'):
        tags.append(job.get('tagABC'))
    return unique_join(tags)


## 单元格 11：把单个岗位整理成一行

原始 JSON 很深，不方便直接看。本单元格把一个岗位字典整理成表格中的一行，同时保留 `原始JSON`，避免后续筛字段时丢信息。


In [10]:
# 单元格 11：单岗位字段整理
# 说明：这里是字段取舍最集中的地方。想新增列，可以照着 return 字典继续加。


def flatten_job(job: Dict[str, Any], page: int, keyword: str, city: str, city_code: str) -> Dict[str, Any]:
    """
    Input:
    - job: 单条岗位原始 JSON。
    - page: 请求页码。
    - keyword: 搜索关键词，也是后续批次筛选条件。
    - city: 搜索城市名称。
    - city_code: 智联城市代码；为空时从城市映射表查找。
    Output:
    - dict，岗位字段字典。
    Function:
    - 把单条岗位 JSON 展平成一行表格记录。
    """
    jd = job.get('jobDetailData') or {}
    pos = jd.get('position') or {}
    base = pos.get('base') or {}
    desc = pos.get('desc') or {}
    custom = jd.get('customAttributeInfo') or {}
    workloc = jd.get('workLocation') or {}
    staff_detail = jd.get('staff') or {}
    staff_card = job.get('staffCard') or {}
    state = get_path(jd, ['stateInfo', 'state'], {})
    verify_basic = get_path(jd, ['verifyTheTruth', 'basic'], {})
    card = parse_json_field(job.get('cardCustomJson'))

    skill_tags = []
    skill_tags += list_values(job.get('jobSkillTags'), ('name',))
    skill_tags += list_values(job.get('skillLabel'), ('value', 'name'))
    skill_tags += list_values(job.get('showSkillTags'), ('tag', 'name', 'value'))

    welfare_tags = []
    welfare_tags += list_values(desc.get('welfareLabel'))
    welfare_tags += list_values(desc.get('welfareTags'))
    welfare_tags += list_values(job.get('welfareLabel'))
    welfare_tags += key_value_items(custom.get('welfareItems'))

    return {
        '搜索关键词': keyword,
        '搜索城市': city,
        '城市代码': city_code,
        '页码': page,
        '职位ID': job.get('jobId') or base.get('positionId'),
        '职位编号': job.get('number') or base.get('positionNumber'),
        '职位名称': job.get('name') or base.get('positionName'),
        '职位URL': job.get('positionURL') or job.get('positionUrl') or base.get('positionUrl'),
        '薪资': job.get('salary60') or base.get('salary') or card.get('salary60'),
        '薪资原始区间': job.get('salaryReal') or base.get('salaryReal'),
        '薪资类型': job.get('salaryType', ''),
        '薪资发放次数': job.get('salaryCount', ''),
        '工作城市': job.get('workCity') or workloc.get('positionWorkCity'),
        '行政区': job.get('cityDistrict') or workloc.get('positionCityDistrict'),
        '商圈/街道': unique_join([job.get('tradingArea'), job.get('streetName'), workloc.get('tradingArea'), workloc.get('streetName')]),
        '工作地点展示': workloc.get('address') or card.get('address') or job.get('workCity'),
        '详细地址': workloc.get('workAddress', ''),
        '经度': workloc.get('longitude', ''),
        '纬度': workloc.get('latitude', ''),
        '经验要求': job.get('workingExp') or base.get('positionWorkingExp'),
        '学历要求': job.get('education') or base.get('education'),
        '工作类型': job.get('workType') or base.get('workType'),
        '工作模式': job.get('workMode') or state.get('workModeDesc') or state.get('workMode'),
        '职位类别': unique_join([
            job.get('jobTypeLevelName'),
            job.get('subJobTypeLevelName'),
            get_path(jd, ['jobType', 'jobTypeLevelName']),
            get_path(jd, ['jobType', 'subJobTypeLevelName']),
        ]),
        '公司名称': job.get('companyName') or card.get('companyName'),
        '公司编号': job.get('companyNumber'),
        '公司URL': job.get('companyUrl'),
        '公司Logo': job.get('companyLogo'),
        '公司规模': job.get('companySize'),
        '公司性质': job.get('propertyName') or job.get('property'),
        '融资阶段': job.get('financingStage') or card.get('strengthLabel'),
        '行业': job.get('industryName'),
        '发布时间': job.get('publishTime') or get_path(pos, ['date', 'positionPublishTime']) or get_path(pos, ['date', 'positionUpdateTimeText']),
        '首次发布时间': job.get('firstPublishTime') or get_path(pos, ['date', 'firstPublishTime']),
        '发布日期文本': get_path(pos, ['date', 'positionUpdateTimeText']),
        '是否新职位': job.get('isNewPosition'),
        '招聘人数': job.get('recruitNumber') or base.get('recruitNumber'),
        'HR姓名': staff_card.get('staffName') or staff_detail.get('staffName'),
        'HR职位': staff_card.get('hrJob') or staff_detail.get('hrJob'),
        'HR状态': staff_card.get('hrStateInfo') or staff_detail.get('hrStateInfo'),
        'HR活跃标签': unique_join(staff_detail.get('activityLevel') or []),
        '职位标签汇总': collect_all_tags(job),
        '技能标签': unique_join(skill_tags),
        '搜索命中关键词': unique_join(list_values(get_path(job, ['jobKeyword', 'keywords'], []), ('itemValue', 'name', 'value'))),
        '福利标签': unique_join(welfare_tags),
        '福利明细': unique_join(key_value_items(custom.get('welfareItems'))),
        '工作时间': unique_join(key_value_items(custom.get('workTimeItems'))),
        '报告项/保障项': unique_join(list_values(custom.get('reportItems'))),
        '职位描述': desc.get('description') or get_path(pos, ['desc', 'description']),
        '职位亮点': desc.get('descriptionHighlight') or job.get('positionHighlight'),
        '职位摘要': job.get('jobSummary', ''),
        '认证/守护信息': verify_basic.get('description') or get_path(jd, ['secure', 'safeCenter', 'title']),
        '原始JSON': json.dumps(job, ensure_ascii=False),
    }


## 单元格 12：主爬取函数

这一格负责翻页、请求 JSON 接口、解析职位列表，并返回三个 DataFrame：`clean_df` 是最终导出的整理字段；`raw_df` 和 `meta_df` 只用于检查原始字段和每页抓取情况，不会在导出单元格里保存成文件。


In [11]:
# 单元格 12：主爬取函数
# 返回值说明：clean_df 是最终导出的整理表；raw_df 和 meta_df 只用于检查字段和页码抓取情况，不再导出文件。


def crawl_zhaopin(
    keyword: str,
    city: str,
    city_code: Optional[str] = None,
    start_page: int = 1,
    max_pages: int = 1,
    page_size: int = 20,
    extra_params: Optional[Dict[str, Any]] = None,
    empty_page_stop: int = 2,
    show_page_progress: bool = True,
):
    """
    Input:
    - keyword: 搜索关键词，也是后续批次筛选条件。
    - city: 搜索城市名称。
    - city_code: 智联城市代码；为空时从城市映射表查找。
    - start_page: 开始抓取的页码。
    - max_pages: 最多抓取页数。
    - page_size: 每页岗位数量。
    - extra_params: 额外接口筛选参数。
    - empty_page_stop: 连续空页达到该数量后停止当前城市。
    - show_page_progress: 是否显示单城市页码进度条。
    Output:
    - tuple，clean_df、raw_df、meta_df。
    Function:
    - 按关键词、城市和页码范围抓取岗位数据。
    """
    resolved_city_code = resolve_city_code(city, city_code)
    all_jobs = []
    meta_rows = []
    stop_page = start_page + max_pages - 1
    empty_page_count = 0
    request_session = make_request_session()

    try:
        page_range = range(start_page, stop_page + 1)
        if show_page_progress:
            page_range = tqdm(page_range, desc=f'{city}页码', leave=False)

        for page_no in page_range:
            page_url = build_search_page_url(keyword, resolved_city_code, page_no, extra_params=extra_params)
            print(f'[{city}] 抓取第 {page_no} 页：{page_url}')

            try:
                data, payload, api_url = fetch_position_page(
                    keyword=keyword,
                    city_code=resolved_city_code,
                    page=page_no,
                    page_size=page_size,
                    extra_params=extra_params,
                    timeout=REQUEST_TIMEOUT,
                    request_session=request_session,
                )
            except PageRequestError as exc:
                failed_payload = build_search_payload(keyword, resolved_city_code, page_no, page_size, extra_params)
                meta_rows.append({
                    'page': page_no,
                    'url': page_url,
                    'api_url': SEARCH_API_URL,
                    'jobs': 0,
                    'reported_pages': '',
                    'position_count': '',
                    'is_end_page': '',
                    'is_verification': '',
                    'task_id': '',
                    'error': repr(exc),
                    'request_payload': json.dumps(failed_payload, ensure_ascii=False),
                })
                print(f'[{city}] 第 {page_no} 页多次重试仍失败，保留当前城市已抓到的数据并停止该城市：{exc!r}')
                break

            jobs, position_count, is_end_page = extract_position_page_data(data)
            reported_pages = (position_count + page_size - 1) // page_size if position_count else 0

            meta_rows.append({
                'page': page_no,
                'url': page_url,
                'api_url': api_url,
                'jobs': len(jobs),
                'reported_pages': reported_pages,
                'position_count': position_count,
                'is_end_page': int(is_end_page),
                'is_verification': data.get('isVerification', ''),
                'task_id': data.get('taskId', ''),
                'error': '',
                'request_payload': json.dumps(payload, ensure_ascii=False),
            })

            if not jobs:
                empty_page_count += 1
                print(f'[{city}] 当前页没有职位，连续空页数：{empty_page_count}/{empty_page_stop}')
                if empty_page_count >= empty_page_stop:
                    print(f'[{city}] 连续空页达到停止阈值，停止抓取。')
                    break
                if page_no < stop_page:
                    time.sleep(random.uniform(*PAGE_SLEEP_RANGE))
                continue

            empty_page_count = 0
            for job in jobs:
                item = dict(job)
                item['_page'] = page_no
                all_jobs.append(item)

            if is_end_page:
                print(f'[{city}] 接口提示已经到达最后一页。')
                break

            if page_no < stop_page:
                time.sleep(random.uniform(*PAGE_SLEEP_RANGE))
    finally:
        request_session.close()

    clean_rows = [
        flatten_job(job, page=job.get('_page', ''), keyword=keyword, city=city, city_code=resolved_city_code)
        for job in all_jobs
    ]
    clean_df = pd.DataFrame(clean_rows)
    raw_df = pd.json_normalize(all_jobs, sep='.') if all_jobs else pd.DataFrame()
    meta_df = pd.DataFrame(meta_rows)
    return clean_df, raw_df, meta_df

## 单元格 13：执行抓取并自动入库

运行这一格才会真正访问接口。当前配置会按 CITY_LIST 批量抓取网页上方城市入口中的每个城市，每个城市抓取 START_PAGE 到 START_PAGE + MAX_PAGES - 1 页。

入库规则：

- CITY_LIST 只决定本次抓取哪些城市，不再生成额外的数据范围字段。
- 本地不再保存 CSV，爬虫结果会拆分写入原始岗位、原始公司、原始 HR 和爬虫日志表。
- 同一个 keyword 最多保留最近 5 个 run_id 批次，超过后自动删除最老批次的业务数据。


In [12]:
# 单元格 13：执行抓取并自动入库
# 说明：爬虫结果直接写入 MySQL 原始分表，不再保存本地 CSV。


def safe_run_text(text):
    """
    Input:
    - text: 需要判断、清理或解析的文本。
    Output:
    - str，安全文本。
    Function:
    - 清理文本，使其适合写入批次编号。
    """
    return re.sub(r'[\s\\/:*?"<>|]+', '_', str(text)).strip('_')


CRAWL_START_TIME = datetime.now()  # 本次爬虫开始时间，用于写入批次日志

# 原始岗位表字段顺序：DataFrame 写入 MySQL 前会按这个顺序补齐和排序。
RAW_JOB_COLUMNS = [
    'run_id', 'batch_no', 'keyword', 'source_city', 'city_code', 'page_no', 'job_sign', 'is_current', 'created_at',
    '职位ID', '职位编号', '职位名称', '职位URL', '薪资', '薪资原始区间', '薪资类型', '薪资发放次数', '工作城市', '行政区',
    '商圈/街道', '工作地点展示', '详细地址', '经度', '纬度', '经验要求', '学历要求', '工作类型', '工作模式', '职位类别',
    '公司名称', '发布时间', '首次发布时间', '发布日期文本', '是否新职位', '招聘人数', '职位标签汇总', '搜索命中关键词',
    '技能标签', '福利标签', '福利明细', '工作时间', '报告项/保障项', '职位描述', '职位亮点', '职位摘要', '认证/守护信息', '原始JSON'
]
# 原始公司表字段顺序：从岗位结果中抽取公司信息后写入 raw_company_info。
RAW_COMPANY_COLUMNS = [
    'run_id', 'batch_no', 'keyword', 'source_city', 'company_sign', 'is_current', 'created_at',
    '公司名称', '公司编号', '公司URL', '公司Logo', '公司规模', '公司性质', '融资阶段', '行业'
]
# 原始 HR 表字段顺序：从岗位结果中抽取 HR 状态信息后写入 raw_hr_status_info。
RAW_HR_COLUMNS = [
    'run_id', 'batch_no', 'keyword', 'source_city', 'job_sign', 'hr_sign', 'is_current', 'created_at',
    '职位名称', '公司名称', 'HR姓名', 'HR职位', 'HR状态', 'HR活跃标签'
]


def make_mysql_engine():
    """
    Input:
    - 无。
    Output:
    - Engine，MySQL 连接引擎。
    Function:
    - 创建 SQLAlchemy MySQL 连接引擎。
    """
    password = quote_plus(MYSQL_PASSWORD)
    url = f'mysql+pymysql://{MYSQL_USER}:{password}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}?charset=utf8mb4'
    return create_engine(url, pool_pre_ping=True)


def stable_hash(*values):
    """
    Input:
    - *values: 参与拼接或签名的一组值。
    Output:
    - str，MD5 签名。
    Function:
    - 根据多个字段生成稳定 MD5 签名。
    """
    parts = []
    for value in values:
        if pd.isna(value):
            parts.append('')
        else:
            parts.append(str(value).strip())
    return hashlib.md5('|'.join(parts).encode('utf-8')).hexdigest()


def build_job_sign(row):
    """
    Input:
    - row: DataFrame 中的一行岗位数据。
    Output:
    - str，岗位签名。
    Function:
    - 生成岗位签名。
    """
    return stable_hash(
        row.get('职位ID', ''),
        row.get('职位名称', ''),
        row.get('公司名称', ''),
        row.get('工作地点展示', ''),
        row.get('发布时间', ''),
    )


def next_batch_no(connection):
    """
    Input:
    - connection: 数据库连接对象。
    Output:
    - int，下一批次号。
    Function:
    - 查询当前关键词的下一个批次号。
    """
    result = connection.execute(
        text(f'SELECT COALESCE(MAX(batch_no), 0) + 1 FROM {CRAWLER_LOG_TABLE} WHERE keyword = :keyword'),
        {'keyword': KEYWORD},
    ).scalar_one()
    return int(result)


def make_run_id(batch_no):
    """
    Input:
    - batch_no: 当前批次号。
    Output:
    - str，run_id。
    Function:
    - 生成可读批次唯一编号。
    """
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    return f'{stamp}_{safe_run_text(KEYWORD)}_batch{batch_no}'


def normalize_db_frame(df, columns):
    """
    Input:
    - df: 待处理的 DataFrame。
    - columns: 目标列名列表和列顺序。
    Output:
    - DataFrame，按目标列排序的数据。
    Function:
    - 补齐数据库目标列并把 NaN 转为 None。
    """
    for column in columns:
        if column not in df.columns:
            df[column] = None
    return df[columns].where(pd.notna(df[columns]), None)


def add_common_batch_columns(df, run_id, batch_no, created_at):
    """
    Input:
    - df: 待处理的 DataFrame。
    - run_id: 批次唯一编号。
    - batch_no: 当前批次号。
    - created_at: 本批次写入时间。
    Output:
    - DataFrame，带批次字段的数据。
    Function:
    - 为爬虫结果补充批次管理字段。
    """
    df = df.copy()
    df['run_id'] = run_id
    df['batch_no'] = batch_no
    df['keyword'] = KEYWORD
    df['source_city'] = df['搜索城市'] if '搜索城市' in df.columns else ''
    df['is_current'] = 1
    df['created_at'] = created_at
    return df


def prepare_raw_job_df(clean_df, run_id, batch_no, created_at):
    """
    Input:
    - clean_df: 爬虫整理后的岗位 DataFrame。
    - run_id: 批次唯一编号。
    - batch_no: 当前批次号。
    - created_at: 本批次写入时间。
    Output:
    - DataFrame，raw_job_info 写入数据。
    Function:
    - 整理原始岗位表写入数据。
    """
    df = add_common_batch_columns(clean_df, run_id, batch_no, created_at)
    df['city_code'] = df['城市代码'] if '城市代码' in df.columns else ''
    page_values = pd.to_numeric(df['页码'], errors='coerce') if '页码' in df.columns else pd.Series([None] * len(df))
    df['page_no'] = page_values.astype('Int64').astype(object).where(page_values.notna(), None)
    df['job_sign'] = df.apply(build_job_sign, axis=1)
    return normalize_db_frame(df, RAW_JOB_COLUMNS)


def prepare_raw_company_df(clean_df, run_id, batch_no, created_at):
    """
    Input:
    - clean_df: 爬虫整理后的岗位 DataFrame。
    - run_id: 批次唯一编号。
    - batch_no: 当前批次号。
    - created_at: 本批次写入时间。
    Output:
    - DataFrame，raw_company_info 写入数据。
    Function:
    - 整理原始公司表写入数据。
    """
    df = add_common_batch_columns(clean_df, run_id, batch_no, created_at)
    df['company_sign'] = df.apply(
        lambda row: stable_hash(KEYWORD, row.get('source_city', ''), row.get('公司名称', '')),
        axis=1,
    )
    df = normalize_db_frame(df, RAW_COMPANY_COLUMNS)
    return df.drop_duplicates(subset=['run_id', 'keyword', 'source_city', '公司名称'], keep='first')


def prepare_raw_hr_df(clean_df, run_id, batch_no, created_at):
    """
    Input:
    - clean_df: 爬虫整理后的岗位 DataFrame。
    - run_id: 批次唯一编号。
    - batch_no: 当前批次号。
    - created_at: 本批次写入时间。
    Output:
    - DataFrame，raw_hr_status_info 写入数据。
    Function:
    - 整理原始 HR 表写入数据。
    """
    df = add_common_batch_columns(clean_df, run_id, batch_no, created_at)
    df['job_sign'] = df.apply(build_job_sign, axis=1)
    df['hr_sign'] = df.apply(
        lambda row: stable_hash(row.get('job_sign', ''), row.get('HR姓名', ''), row.get('HR职位', ''), row.get('HR状态', ''), row.get('HR活跃标签', '')),
        axis=1,
    )
    df = normalize_db_frame(df, RAW_HR_COLUMNS)
    return df.drop_duplicates(subset=['run_id', 'job_sign', 'hr_sign'], keep='first')


def set_previous_batches_not_current(connection):
    """
    Input:
    - connection: 数据库连接对象。
    Output:
    - None。
    Function:
    - 把当前关键词旧批次标记为非当前。
    """
    params = {'keyword': KEYWORD}
    for table in [RAW_JOB_TABLE, RAW_COMPANY_TABLE, RAW_HR_TABLE]:
        connection.execute(
            text(f'UPDATE {table} SET is_current = 0 WHERE keyword = :keyword'),
            params,
        )
    connection.execute(
        text(f'UPDATE {CRAWLER_LOG_TABLE} SET is_current = 0 WHERE keyword = :keyword'),
        params,
    )


def enforce_raw_retention(connection):
    """
    Input:
    - connection: 数据库连接对象。
    Output:
    - list[str]，被删除的 run_id 列表。
    Function:
    - 执行原始层最近批次保留策略。
    """
    rows = connection.execute(
        text(f'''
            SELECT run_id
            FROM {CRAWLER_LOG_TABLE}
            WHERE keyword = :keyword AND is_retained = 1
            ORDER BY batch_no DESC, log_id DESC
        '''),
        {'keyword': KEYWORD},
    ).fetchall()
    old_run_ids = [row[0] for row in rows[KEEP_RECENT_BATCHES:]]
    for old_run_id in old_run_ids:
        for table in [RAW_JOB_TABLE, RAW_COMPANY_TABLE, RAW_HR_TABLE]:
            connection.execute(text(f'DELETE FROM {table} WHERE run_id = :run_id'), {'run_id': old_run_id})
        connection.execute(
            text(f'''
                UPDATE {CRAWLER_LOG_TABLE}
                SET is_retained = 0, is_current = 0, deleted_at = NOW()
                WHERE run_id = :run_id
            '''),
            {'run_id': old_run_id},
        )
    return old_run_ids


def insert_crawler_log(connection, run_id, batch_no, job_count, company_count, hr_count, summary_df, failed_df, start_time, end_time):
    """
    Input:
    - connection: 数据库连接对象。
    - run_id: 批次唯一编号。
    - batch_no: 当前批次号。
    - job_count: 本批次岗位写入记录数。
    - company_count: 本批次公司写入记录数。
    - hr_count: 本批次 HR 写入记录数。
    - summary_df: 城市抓取汇总 DataFrame。
    - failed_df: 失败城市 DataFrame。
    - start_time: 任务开始时间。
    - end_time: 任务结束时间。
    Output:
    - tuple，运行状态和运行说明。
    Function:
    - 写入爬虫批次日志。
    """
    planned_city_count = len(CITY_LIST)
    failed_city_count = len(failed_df) if failed_df is not None and not failed_df.empty else 0
    success_city_count = max(0, planned_city_count - failed_city_count)
    if job_count == 0:
        run_status = 'failed'
    elif failed_city_count:
        run_status = 'partial'
    else:
        run_status = 'success'

    if failed_city_count:
        failed_names = ','.join(failed_df['城市'].dropna().astype(str).tolist()) if '城市' in failed_df.columns else ''
        run_message = f'失败城市: {failed_names}'
    else:
        run_message = '爬虫入库完成'

    connection.execute(
        text(f'''
            INSERT INTO {CRAWLER_LOG_TABLE}
            (run_id, batch_no, keyword, source_cities, planned_city_count, success_city_count,
             failed_city_count, job_record_count, company_record_count, hr_record_count, run_status, run_message,
             start_time, end_time, is_current, is_retained, created_at)
            VALUES
            (:run_id, :batch_no, :keyword, :source_cities, :planned_city_count, :success_city_count,
             :failed_city_count, :job_record_count, :company_record_count, :hr_record_count, :run_status, :run_message,
             :start_time, :end_time, :is_current, 1, :created_at)
        '''),
        {
            'run_id': run_id,
            'batch_no': batch_no,
            'keyword': KEYWORD,
            'source_cities': ','.join(CITY_LIST),
            'planned_city_count': planned_city_count,
            'success_city_count': success_city_count,
            'failed_city_count': failed_city_count,
            'job_record_count': int(job_count),
            'company_record_count': int(company_count),
            'hr_record_count': int(hr_count),
            'run_status': run_status,
            'run_message': run_message[:1000],
            'start_time': start_time.strftime('%Y-%m-%d %H:%M:%S'),
            'end_time': end_time.strftime('%Y-%m-%d %H:%M:%S'),
            'is_current': 1 if job_count else 0,
            'created_at': end_time.strftime('%Y-%m-%d %H:%M:%S'),
        },
    )
    return run_status, run_message


def save_crawl_result_to_mysql(clean_df, summary_df, failed_df, start_time):
    """
    Input:
    - clean_df: 爬虫整理后的岗位 DataFrame。
    - summary_df: 城市抓取汇总 DataFrame。
    - failed_df: 失败城市 DataFrame。
    - start_time: 任务开始时间。
    Output:
    - dict，批次信息和写入统计。
    Function:
    - 把爬虫结果拆分写入 MySQL 原始层。
    """
    engine = make_mysql_engine()
    end_time = datetime.now()
    created_at = end_time.strftime('%Y-%m-%d %H:%M:%S')

    with engine.begin() as connection:
        batch_no = next_batch_no(connection)
        run_id = make_run_id(batch_no)

        if clean_df.empty:
            insert_crawler_log(connection, run_id, batch_no, 0, 0, 0, summary_df, failed_df, start_time, end_time)
            return {
                'run_id': run_id,
                'batch_no': batch_no,
                'job_count': 0,
                'company_count': 0,
                'hr_count': 0,
                'deleted_run_ids': [],
            }

        raw_job_df = prepare_raw_job_df(clean_df, run_id, batch_no, created_at)
        raw_company_df = prepare_raw_company_df(clean_df, run_id, batch_no, created_at)
        raw_hr_df = prepare_raw_hr_df(clean_df, run_id, batch_no, created_at)

        set_previous_batches_not_current(connection)
        raw_job_df.to_sql(RAW_JOB_TABLE, con=connection, if_exists='append', index=False, chunksize=300, method='multi')
        raw_company_df.to_sql(RAW_COMPANY_TABLE, con=connection, if_exists='append', index=False, chunksize=300, method='multi')
        raw_hr_df.to_sql(RAW_HR_TABLE, con=connection, if_exists='append', index=False, chunksize=300, method='multi')
        insert_crawler_log(connection, run_id, batch_no, len(raw_job_df), len(raw_company_df), len(raw_hr_df), summary_df, failed_df, start_time, end_time)
        deleted_run_ids = enforce_raw_retention(connection)

    return {
        'run_id': run_id,
        'batch_no': batch_no,
        'job_count': len(raw_job_df) if not clean_df.empty else 0,
        'company_count': len(raw_company_df) if not clean_df.empty else 0,
        'hr_count': len(raw_hr_df) if not clean_df.empty else 0,
        'deleted_run_ids': deleted_run_ids,
    }


WORKER_COUNT = min(max(1, int(MAX_CITY_WORKERS)), len(CITY_LIST)) if CITY_LIST else 1  # 实际城市并发数，不超过城市数量
city_clean_frames = []  # 保存每个城市抓到的岗位明细 DataFrame
city_summary_rows = []  # 保存每个城市的抓取汇总记录
failed_cities = []  # 保存失败城市和错误信息


def crawl_city_task(city_index: int, city_name: str):
    """
    Input:
    - city_index: 城市在 CITY_LIST 中的序号。
    - city_name: 当前抓取的城市名称。
    Output:
    - dict，单城市抓取结果。
    Function:
    - 抓取单个城市并返回结果与汇总。
    """
    print(f'\n===== 开始抓取城市：{city_name}，页码 {START_PAGE}-{START_PAGE + MAX_PAGES - 1} =====')
    try:
        current_clean_df, _, current_meta_df = crawl_zhaopin(
            keyword=KEYWORD,
            city=city_name,
            city_code=None,
            start_page=START_PAGE,
            max_pages=MAX_PAGES,
            page_size=PAGE_SIZE,
            extra_params=EXTRA_PARAMS,
            empty_page_stop=EMPTY_PAGE_STOP,
            show_page_progress=(WORKER_COUNT == 1),
        )
    except Exception as exc:
        error_text = repr(exc)
        print(f'城市 {city_name} 抓取失败：{error_text}')
        return {
            'index': city_index,
            'city': city_name,
            'clean_df': pd.DataFrame(),
            'summary': {'城市': city_name, '抓取页数': 0, '记录数': 0, '状态': '失败', '错误': error_text},
            'failure': {'城市': city_name, '错误': error_text},
        }

    page_count = int(current_meta_df['page'].nunique()) if not current_meta_df.empty and 'page' in current_meta_df.columns else 0
    row_count = len(current_clean_df)
    error_messages = []
    if not current_meta_df.empty and 'error' in current_meta_df.columns:
        error_messages = [msg for msg in current_meta_df['error'].dropna().astype(str).tolist() if msg]
    status = '成功'
    if error_messages:
        status = '部分成功' if row_count else '失败'

    if current_clean_df.empty:
        print(f'城市 {city_name} 没有抓到数据。')
    else:
        print(f'城市 {city_name} 已整理：{row_count} 条')

    return {
        'index': city_index,
        'city': city_name,
        'clean_df': current_clean_df,
        'summary': {
            '城市': city_name,
            '抓取页数': page_count,
            '记录数': row_count,
            '状态': status,
            '错误': error_messages[-1] if error_messages else '',
        },
        'failure': {'城市': city_name, '错误': error_messages[-1]} if status == '失败' and error_messages else None,
    }


print(f'城市并发数：{WORKER_COUNT}')
print('MySQL原始表:', f'{MYSQL_DATABASE}.{RAW_JOB_TABLE}, {RAW_COMPANY_TABLE}, {RAW_HR_TABLE}')
city_results = []
if WORKER_COUNT == 1:
    for index, city_name in enumerate(tqdm(CITY_LIST, desc='抓取城市')):
        city_results.append(crawl_city_task(index, city_name))
else:
    with ThreadPoolExecutor(max_workers=WORKER_COUNT) as executor:
        future_to_city = {
            executor.submit(crawl_city_task, index, city_name): city_name
            for index, city_name in enumerate(CITY_LIST)
        }
        for future in tqdm(as_completed(future_to_city), total=len(future_to_city), desc='抓取城市'):
            city_results.append(future.result())

city_results.sort(key=lambda item: item['index'])
for result in city_results:
    city_summary_rows.append(result['summary'])
    if result['failure']:
        failed_cities.append(result['failure'])
    if not result['clean_df'].empty:
        city_clean_frames.append(result['clean_df'])

clean_df = pd.concat(city_clean_frames, ignore_index=True) if city_clean_frames else pd.DataFrame()
summary_df = pd.DataFrame(city_summary_rows)
failed_df = pd.DataFrame(failed_cities)
db_result = save_crawl_result_to_mysql(clean_df, summary_df, failed_df, CRAWL_START_TIME)

print('\n===== 抓取完成 =====')
print(f'计划城市数：{len(CITY_LIST)}')
print(f'整理后总记录数：{len(clean_df)}')
print(f'批次号 batch_no：{db_result["batch_no"]}')
print(f'批次ID run_id：{db_result["run_id"]}')
print(f'岗位入库记录数：{db_result["job_count"]}')
print(f'公司入库记录数：{db_result["company_count"]}')
print(f'HR入库记录数：{db_result["hr_count"]}')
if db_result['deleted_run_ids']:
    print('超过最近5批，已删除最老原始业务批次:', ', '.join(db_result['deleted_run_ids']))

if not summary_df.empty:
    display(summary_df)
if not failed_df.empty:
    print('以下城市抓取失败：')
    display(failed_df)


城市并发数：4
MySQL原始表: shixun.raw_job_info, raw_company_info, raw_hr_status_info

===== 开始抓取城市：北京，页码 1-5 =====
[北京] 抓取第 1 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=530&order=4

===== 开始抓取城市：上海，页码 1-5 =====
[上海] 抓取第 1 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=538&order=4

===== 开始抓取城市：广州，页码 1-5 =====
[广州] 抓取第 1 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=763&order=4

===== 开始抓取城市：深圳，页码 1-5 =====
[深圳] 抓取第 1 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=765&order=4


抓取城市:   0%|          | 0/24 [00:00<?, ?it/s]

[北京] 抓取第 2 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=2&jl=530&order=4
[上海] 抓取第 2 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=2&jl=538&order=4
[深圳] 抓取第 2 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=2&jl=765&order=4
[广州] 抓取第 2 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=2&jl=763&order=4
[北京] 抓取第 3 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=3&jl=530&order=4
[上海] 抓取第 3 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=3&jl=538&order=4
[深圳] 抓取第 3 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=3&jl=765&order=4
[广州] 抓取第 3 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=3&jl=763&order=4
[上海] 抓取第 4 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=4&jl=538&order=4
[北京] 抓取第 4 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=4&jl=53

抓取城市:   8%|▊         | 2/24 [00:04<00:42,  1.92s/it]

城市 北京 已整理：100 条

===== 开始抓取城市：武汉，页码 1-5 =====
[武汉] 抓取第 1 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=736&order=4
城市 深圳 已整理：100 条

===== 开始抓取城市：西安，页码 1-5 =====
[西安] 抓取第 1 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=854&order=4
[广州] 抓取第 5 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=5&jl=763&order=4


...

[哈尔滨] 接口提示已经到达最后一页。
城市 哈尔滨 已整理：40 条
[厦门] 抓取第 3 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=3&jl=682&order=4


抓取城市: 100%|██████████| 24/24 [00:24<00:00,  1.02s/it]

[厦门] 接口提示已经到达最后一页。
城市 厦门 已整理：45 条



===== 抓取完成 =====
计划城市数：24
整理后总记录数：1822
批次号 batch_no：1
批次ID run_id：20260618_082950_数据开发_batch1
岗位入库记录数：1822
公司入库记录数：1276
HR入库记录数：1822


,城市,抓取页数,记录数,状态,错误
0,北京,5,100,成功,
1,上海,5,100,成功,
2,广州,5,100,成功,
3,深圳,5,100,成功,
4,天津,5,91,成功,
5,武汉,5,100,成功,
6,西安,5,100,成功,
7,成都,5,100,成功,
8,大连,3,44,成功,
9,长春,2,39,成功,
